In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import plotly.graph_objects as go
import logging
import ast
import math
from collections import Counter

from torch.nn.utils.rnn import pad_sequence

import torch
from torch import nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

import utils
import z_utils

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = "cpu"
print(device)

cuda


# 1. Data

In [3]:
spike_df = pd.read_csv('../../Data_processed/spike_tensors_x.csv')

spike_df['spike_durations'] = spike_df['spike_durations'].apply(ast.literal_eval)
spike_df['spike_durations'] = spike_df['spike_durations'].apply(lambda x: x + [0]*(4-len(x)))

spike_df['spike_magnitudes'] = spike_df['spike_magnitudes'].apply(ast.literal_eval)
spike_df['spike_magnitudes'] = spike_df['spike_magnitudes'].apply(lambda x: x + [0.0]*(4-len(x)))

spike_df['time'] = spike_df['spike_times_intervals'].apply(ast.literal_eval)
spike_df['time'] = spike_df['time'].apply(lambda x: x + [0]*(4-len(x)))

print(spike_df.head())

   spike_num    spike_type spike_durations              spike_magnitudes  \
0          3     [2, 2, 3]    [3, 3, 4, 0]    [0.663, 0.493, 0.886, 0.0]   
1          2     [3, 2, 3]    [4, 5, 0, 0]      [0.933, 1.306, 0.0, 0.0]   
2          2     [2, 2, 3]    [6, 9, 0, 0]      [1.191, 1.085, 0.0, 0.0]   
3          4  [3, 2, 2, 3]    [3, 3, 3, 3]  [1.075, 0.551, 0.695, 1.164]   
4          3  [2, 3, 2, 3]    [3, 3, 5, 0]    [0.408, 0.991, 1.413, 0.0]   

                                       ID-statistics  \
0  [0.2525107493475829, 0.158, 0.2470795043314368...   
1  [0.2525107493475829, 0.158, 0.2470795043314368...   
2  [0.2525107493475829, 0.158, 0.2470795043314368...   
3  [0.2525107493475829, 0.158, 0.2470795043314368...   
4  [0.2525107493475829, 0.158, 0.2470795043314368...   

                                             weather spike_times_intervals  \
0  [0.44636145833333335, 0.3711697916666667, 0.66...          [24, 34, 38]   
1  [0.37326145833333335, 0.1712510416666667, 0.80.

In [4]:
# Get the probability of different spike num
spike_num = spike_df['spike_num'].value_counts()
spike_num = spike_num.sort_index()
total = spike_num.sum()
spike_num = spike_num / total
prob_df = pd.DataFrame(spike_num)
prob_df.columns = ['probability']
prob_df= prob_df.reset_index()
prob_df = prob_df.sort_values(by='probability', ascending=False)

print(prob_df)


    spike_num  probability
2           2     0.305778
3           3     0.269052
1           1     0.157257
4           4     0.133918
0           0     0.083342
5           5     0.040030
6           6     0.008242
7           7     0.001684
8           8     0.000406
9           9     0.000167
10         10     0.000086
11         11     0.000032
12         12     0.000007


In [5]:
# Only keep the days with spike_num < 5
spike_df = spike_df[spike_df['spike_num'] < 5]
spike_num = spike_df['spike_num'].value_counts()
spike_num = spike_num.sort_index()
total = spike_num.sum()
spike_num = spike_num / total
prob_df = pd.DataFrame(spike_num)
prob_df.columns = ['probability']
prob_df= prob_df.reset_index()
prob_df = prob_df.sort_values(by='probability', ascending=False)

print(prob_df)


   spike_num  probability
2          2     0.322093
3          3     0.283407
1          1     0.165647
4          4     0.141064
0          0     0.087789


In [6]:
all_durations = [duration for sublist in spike_df['spike_durations'] for duration in sublist if duration != 0]

# Count the occurrences of each duration
duration_counts = Counter(all_durations)

# Calculate the total number of non-zero durations
total_durations = sum(duration_counts.values())

# Calculate the probability of each duration
duration_probabilities = {duration: count / total_durations for duration, count in duration_counts.items()}

# Convert to a DataFrame for better readability
probability_df = pd.DataFrame(list(duration_probabilities.items()), columns=['Duration', 'Probability'])

# Sort by Duration
probability_df = probability_df.sort_values(by='Probability', ascending=False).reset_index(drop=True)

print(probability_df)

# Add together the probabilities for 10 most common durations
top_10_probability = probability_df['Probability'].iloc[:13].sum()
print(f'The 13 most common durations account for {top_10_probability:.2%} of all non-zero durations')


    Duration   Probability
0          3  4.396302e-01
1          5  1.011166e-01
2          4  8.101210e-02
3          6  7.845331e-02
4          7  5.036503e-02
5          8  4.227612e-02
6          9  3.424870e-02
7         10  2.768852e-02
8         11  2.366418e-02
9         12  1.988363e-02
10        13  1.642147e-02
11        14  1.374648e-02
12        15  1.110663e-02
13         2  8.598698e-03
14        16  8.511165e-03
15         1  8.457382e-03
16        17  6.150865e-03
17        18  4.768835e-03
18        19  3.672365e-03
19        20  2.994449e-03
20        21  2.450606e-03
21        22  2.064415e-03
22        23  1.746647e-03
23        24  1.510093e-03
24        25  1.252272e-03
25        26  1.116042e-03
26        27  9.972252e-04
27        28  9.186307e-04
28        29  8.297111e-04
29        30  7.367846e-04
30        31  6.717516e-04
31        32  5.782087e-04
32        33  4.507624e-04
33        34  3.555244e-04
34        48  3.054397e-04
35        35  2.535057e-04
3

In [7]:
top_durations = set(probability_df.nlargest(7, 'Probability')['Duration'])

# Function to filter and reorder durations in each row
def filter_and_reorder_with_indices(duration_list, magnitude_list, time_list):
    # Ensure the lists are of equal length
    min_length = 4
    
    # Trim the lists if they have inconsistent lengths
    duration_list = duration_list[:min_length]
    magnitude_list = magnitude_list[:min_length]
    time_list = time_list[:min_length]
    
    # Indices of the durations to keep based on the top probabilities
    keep_indices = [i for i, d in enumerate(duration_list) if d in top_durations]

    # Filter and keep only the relevant indices, moving zeros to the end
    filtered_durations = [duration_list[i]-2 for i in keep_indices] + [0] * (min_length - len(keep_indices))
    filtered_magnitudes = [magnitude_list[i] for i in keep_indices] + [0.0] * (min_length - len(keep_indices))
    filtered_time = [time_list[i] for i in keep_indices] + [0] * (min_length - len(keep_indices))

    return filtered_durations, filtered_magnitudes, filtered_time

spike_df[['filtered_spike_durations', 'filtered_spike_magnitudes', 'filtered_time']] = spike_df.apply(
    lambda row: filter_and_reorder_with_indices(row['spike_durations'], row['spike_magnitudes'], row['time']),
    axis=1, result_type='expand'
)

print(spike_df.head())

   spike_num    spike_type spike_durations              spike_magnitudes  \
0          3     [2, 2, 3]    [3, 3, 4, 0]    [0.663, 0.493, 0.886, 0.0]   
1          2     [3, 2, 3]    [4, 5, 0, 0]      [0.933, 1.306, 0.0, 0.0]   
2          2     [2, 2, 3]    [6, 9, 0, 0]      [1.191, 1.085, 0.0, 0.0]   
3          4  [3, 2, 2, 3]    [3, 3, 3, 3]  [1.075, 0.551, 0.695, 1.164]   
4          3  [2, 3, 2, 3]    [3, 3, 5, 0]    [0.408, 0.991, 1.413, 0.0]   

                                       ID-statistics  \
0  [0.2525107493475829, 0.158, 0.2470795043314368...   
1  [0.2525107493475829, 0.158, 0.2470795043314368...   
2  [0.2525107493475829, 0.158, 0.2470795043314368...   
3  [0.2525107493475829, 0.158, 0.2470795043314368...   
4  [0.2525107493475829, 0.158, 0.2470795043314368...   

                                             weather spike_times_intervals  \
0  [0.44636145833333335, 0.3711697916666667, 0.66...          [24, 34, 38]   
1  [0.37326145833333335, 0.1712510416666667, 0.80.

In [8]:
all_durations = [duration for sublist in spike_df['filtered_spike_durations'] for duration in sublist if duration != 0]
# Count the occurrences of each duration
duration_counts = Counter(all_durations)

# Calculate the total number of non-zero durations
total_durations = sum(duration_counts.values())

# Calculate the probability of each duration
duration_probabilities = {duration: count / total_durations for duration, count in duration_counts.items()}

# Convert to a DataFrame for better readability
probability_df = pd.DataFrame(list(duration_probabilities.items()), columns=['Duration', 'Probability'])

# Sort by Duration
probability_df = probability_df.sort_values(by='Probability', ascending=False).reset_index(drop=True)

print(probability_df)

   Duration  Probability
0         1     0.531531
1         3     0.122254
2         2     0.097947
3         4     0.094853
4         5     0.060893
5         6     0.051114
6         7     0.041408


In [9]:
def count_non_zero_durations(duration_list):
    return sum(1 for d in duration_list if d != 0)

# Update the spike_num column to reflect the count of non-zero durations
spike_df['spike_num'] = spike_df['filtered_spike_durations'].apply(count_non_zero_durations)

# Display the first few rows to verify the changes
spike_df.head(20)

,spike_num,spike_type,spike_durations,spike_magnitudes,ID-statistics,weather,spike_times_intervals,date_sin,date_cos,time,filtered_spike_durations,filtered_spike_magnitudes,filtered_time
0,3,"[2, 2, 3]","[3, 3, 4, 0]","[0.663, 0.493, 0.886, 0.0]","[0.2525107493475829, 0.158, 0.2470795043314368...","[0.44636145833333335, 0.3711697916666667, 0.66...","[24, 34, 38]",-0.977848,0.209315,"[24, 34, 38, 0]","[1, 1, 2, 0]","[0.663, 0.493, 0.886, 0.0]","[24, 34, 38, 0]"
1,2,"[3, 2, 3]","[4, 5, 0, 0]","[0.933, 1.306, 0.0, 0.0]","[0.2525107493475829, 0.158, 0.2470795043314368...","[0.37326145833333335, 0.1712510416666667, 0.80...","[20, 36]",-0.974100,0.226116,"[20, 36, 0, 0]","[2, 3, 0, 0]","[0.933, 1.306, 0.0, 0.0]","[20, 36, 0, 0]"
2,2,"[2, 2, 3]","[6, 9, 0, 0]","[1.191, 1.085, 0.0, 0.0]","[0.2525107493475829, 0.158, 0.2470795043314368...","[0.35165416666666666, 0.17661875000000002, 0.7...","[20, 37]",-0.970064,0.242850,"[20, 37, 0, 0]","[4, 7, 0, 0]","[1.191, 1.085, 0.0, 0.0]","[20, 37, 0, 0]"
3,4,"[3, 2, 2, 3]","[3, 3, 3, 3]","[1.075, 0.551, 0.695, 1.164]","[0.2525107493475829, 0.158, 0.2470795043314368...","[0.392646875, 0.22844895833333334, 0.79748125]","[17, 23, 34, 38]",-0.965740,0.259512,"[17, 23, 34, 38]","[1, 1, 1, 1]","[1.075, 0.551, 0.695, 1.164]","[17, 23, 34, 38]"
4,3,"[2, 3, 2, 3]","[3, 3, 5, 0]","[0.408, 0.991, 1.413, 0.0]","[0.2525107493475829, 0.158, 0.2470795043314368...","[0.45284687500000004, 0.3914020833333333, 0.65...","[18, 25, 36]",-0.961130,0.276097,"[18, 25, 36, 0]","[1, 1, 3, 0]","[0.408, 0.991, 1.413, 0.0]","[18, 25, 36, 0]"
5,2,"[2, 2]","[8, 3, 0, 0]","[0.784, 0.377, 0.0, 0.0]","[0.2525107493475829, 0.158, 0.2470795043314368...","[0.49727499999999997, 0.374875, 0.759745833333...","[19, 43]",-0.956235,0.292600,"[19, 43, 0, 0]","[6, 1, 0, 0]","[0.784, 0.377, 0.0, 0.0]","[19, 43, 0, 0]"
6,1,"[3, 2]","[7, 0, 0, 0]","[1.6720000000000002, 0.0, 0.0, 0.0]","[0.2525107493475829, 0.158, 0.2470795043314368...","[0.51195625, 0.274325, 0.8019531249999999]",[38],-0.951057,0.309017,"[38, 0, 0, 0]","[5, 0, 0, 0]","[1.6720000000000002, 0.0, 0.0, 0.0]","[38, 0, 0, 0]"
7,2,"[2, 2]","[3, 3, 0, 0]","[0.406, 0.822, 0.0, 0.0]","[0.2525107493475829, 0.158, 0.2470795043314368...","[0.47235, 0.09946458333333334, 0.9436062500000...","[34, 40]",-0.945596,0.325342,"[34, 40, 0, 0]","[1, 1, 0, 0]","[0.406, 0.822, 0.0, 0.0]","[34, 40, 0, 0]"
8,1,"[2, 3, 2]","[6, 16, 0, 0]","[0.798, 1.879, 0.0, 0.0]","[0.2525107493475829, 0.158, 0.2470795043314368...","[0.45446458333333334, 0.09321770833333333, 0.8...","[22, 33]",-0.939856,0.341571,"[22, 33, 0, 0]","[4, 0, 0, 0]","[0.798, 0.0, 0.0, 0.0]","[22, 0, 0, 0]"
9,2,"[2, 2, 2, 3, 2]","[7, 3, 17, 0]","[1.189, 0.657, 2.6060000000000003, 0.0]","[0.2525107493475829, 0.158, 0.2470795043314368...","[0.4468854166666667, 0.2341895833333333, 0.917...","[20, 28, 32]",-0.933837,0.357698,"[20, 28, 32, 0]","[5, 1, 0, 0]","[1.189, 0.657, 0.0, 0.0]","[20, 28, 0, 0]"


In [10]:
# See the max value for spike_magnitude
max_magnitude = max(spike_df['spike_magnitudes'].apply(max))
print(max_magnitude)

61.311


In [11]:
all_values = [value for tensor in spike_df['filtered_time'] for value in tensor]

# Convert to a set to get the unique values and count them
unique_values = set(all_values)

# Number of unique values
num_unique_values = len(unique_values)
print(unique_values)

# Display the result
print(f"The number of different individual values in 'filtered_time' column is: {num_unique_values}")

{0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46}
The number of different individual values in 'filtered_time' column is: 47


In [12]:
all_values = [value for tensor in spike_df['filtered_spike_durations'] for value in tensor]

# Convert to a set to get the unique values and count them
unique_values = set(all_values)

# Number of unique values
num_unique_values = len(unique_values)

# Display the result
print(f"The number of different individual values in 'spike_durations' column is: {num_unique_values}")

The number of different individual values in 'spike_durations' column is: 8


In [13]:
class SpikeDataset(Dataset):
    def __init__(self, dataframe):
        self.data = dataframe
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        spike_num = torch.tensor(row['spike_num'], dtype=torch.long).unsqueeze(-1)

        spike_durations = torch.tensor(row['filtered_spike_durations'], dtype=torch.long)
        
        spike_magnitudes = torch.tensor(row['filtered_spike_magnitudes'], dtype=torch.float32) / 50
        
        # Ensure that ID-statistics and other fields are properly parsed from strings to lists if necessary
        if isinstance(row['ID-statistics'], str):
            ID_statistics = torch.tensor(ast.literal_eval(row['ID-statistics']), dtype=torch.float32)
        else:
            ID_statistics = torch.tensor(row['ID-statistics'], dtype=torch.float32)

        if isinstance(row['weather'], str):
            weather = torch.tensor(ast.literal_eval(row['weather']), dtype=torch.float32)
        else:
            weather = torch.tensor(row['weather'], dtype=torch.float32)

        date_sin = torch.tensor(row['date_sin'], dtype=torch.float32).unsqueeze(-1)
        date_cos = torch.tensor(row['date_cos'], dtype=torch.float32).unsqueeze(-1)
        
        time = torch.tensor(row['filtered_time'], dtype=torch.long)

        return spike_num, spike_durations, spike_magnitudes, ID_statistics, weather, date_sin, date_cos, time

dataset = SpikeDataset(spike_df)

dataset_size = len(dataset)
train_size = int(0.1 * dataset_size)
val_size = int(0.15 * dataset_size)
test_size = dataset_size - train_size - val_size  # Ensure the sum equals dataset_size

train_ds, val_ds, test_ds = torch.utils.data.random_split(dataset, [train_size, val_size, test_size])

train_dl = DataLoader(train_ds, batch_size=256, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=128, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=128, shuffle=True)
        

In [27]:
import torch
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from torch.utils.data import DataLoader, Dataset
import ast

# Convert DataLoader to numpy arrays for RandomForest (with CUDA support)
def extract_features_and_labels(dataloader):
    features_list = []
    labels_list = []
    
    for spike_num, spike_durations, spike_magnitudes, ID_statistics, weather, date_sin, date_cos, time in dataloader:
        # Move tensors to CPU if they are on CUDA
        ID_statistics = ID_statistics.cpu()
        weather = weather.cpu()
        date_sin = date_sin.cpu()
        date_cos = date_cos.cpu()
        spike_num = spike_num.cpu()

        # Concatenate features (ID_statistics, date_sin, date_cos, weather)
        features = torch.cat([ID_statistics, date_sin, date_cos, weather], dim=1)
        features_list.append(features.numpy())  # Convert to numpy
        labels_list.append(spike_num.squeeze().numpy())  # Labels (spike_num)

    # Combine all batches into a single dataset
    X = np.vstack(features_list)  # Feature matrix
    y = np.concatenate(labels_list)  # Labels
    
    return X, y

# Ensure the device is properly set to 'cuda'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Extract training data
X_train, y_train = extract_features_and_labels(train_dl)

# Extract validation data
X_val, y_val = extract_features_and_labels(val_dl)

# Extract test data
X_test, y_test = extract_features_and_labels(test_dl)

# Initialize the RandomForestClassifier
rf_model = RandomForestClassifier(n_estimators=1000, 
                                  max_depth=20,
                                  min_samples_split=10,
                                  max_features='sqrt',
                                  class_weight='balanced', 
                                  random_state=42)

# Train the model on the training data
rf_model.fit(X_train, y_train)

# Make predictions on validation set
y_val_pred = rf_model.predict(X_val)
val_accuracy = accuracy_score(y_val, y_val_pred)
print(f'Validation Accuracy: {val_accuracy:.4f}')

# Classification report for validation set
print("Validation Classification Report:")
print(classification_report(y_val, y_val_pred, target_names=["0", "1", "2", "3", "4"]))

# Make predictions on test set
y_test_pred = rf_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f'Test Accuracy: {test_accuracy:.4f}')

# Classification report for test set
print("Test Classification Report:")
print(classification_report(y_test, y_test_pred, target_names=["0", "1", "2", "3", "4"]))


Validation Accuracy: 0.3258
Validation Classification Report:
              precision    recall  f1-score   support

           0       0.40      0.45      0.42     68911
           1       0.34      0.33      0.34    107238
           2       0.35      0.26      0.30    127730
           3       0.29      0.32      0.31     92334
           4       0.21      0.33      0.26     41383

    accuracy                           0.33    437596
   macro avg       0.32      0.34      0.33    437596
weighted avg       0.33      0.33      0.33    437596

Test Accuracy: 0.3252
Test Classification Report:
              precision    recall  f1-score   support

           0       0.41      0.45      0.43    345038
           1       0.34      0.33      0.34    537408
           2       0.35      0.26      0.30    635981
           3       0.29      0.32      0.30    461725
           4       0.21      0.33      0.26    207834

    accuracy                           0.33   2187986
   macro avg       

In [14]:
for idx, (spike_nums, spike_durations, spike_magnitudes, ID_statistics, weather, date_sin, date_cos, time) in enumerate(train_dl):
    print('spike_nums shape:', spike_nums.shape)
    print('spike_durations shape:', spike_durations.shape)
    print('spike_magnitudes shape:', spike_magnitudes.shape)
    print('ID_statistics shape:', ID_statistics.shape)
    print('weather shape:', weather.shape)
    print('date_sin shape:', date_sin.shape)
    print('date_cos shape:', date_cos.shape)
    print('time shape:', time.shape)
    break

spike_nums shape: torch.Size([256, 1])
spike_durations shape: torch.Size([256, 4])
spike_magnitudes shape: torch.Size([256, 4])
ID_statistics shape: torch.Size([256, 6])
weather shape: torch.Size([256, 3])
date_sin shape: torch.Size([256, 1])
date_cos shape: torch.Size([256, 1])
time shape: torch.Size([256, 4])


In [ ]:
id = train_ds

# 2. Network

In [15]:
class SpikeNumPredictor(nn.Module):
    def __init__(self):
        super(SpikeNumPredictor, self).__init__()
        # Fully connected layers
        self.fc1 = nn.Linear(11, 64)  # Input size is 11 (ID_statistics + date_sin + date_cos + weather)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 16)
        self.fc4 = nn.Linear(16, 5)   # Output 5 classes (for spike_nums 0-4)
        
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = self.fc4(x)  # No softmax here since CrossEntropyLoss expects raw logits
        return x

In [23]:
model = SpikeNumPredictor().to(device)
criterion = nn.CrossEntropyLoss()  # For classification
optimizer = optim.Adam(model.parameters(), lr=0.01)
epochs = 100

In [24]:
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for spike_nums, spike_durations, spike_magnitudes, ID_statistics, weather, date_sin, date_cos, time in train_dl:

        spike_nums = spike_nums.to(device)
        spike_durations = spike_durations.to(device)
        spike_magnitudes = spike_magnitudes.to(device)
        ID_statistics = ID_statistics.to(device)
        weather = weather.to(device)
        date_sin = date_sin.to(device)
        date_cos = date_cos.to(device)
        time = time.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(torch.cat([ID_statistics, date_sin, date_cos, weather], dim=-1))
        loss = criterion(outputs, spike_nums.squeeze())

        # Backward pass
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    # Validation loss
    val_loss = 0.0
    model.eval()
    with torch.no_grad():
        for spike_nums, spike_durations, spike_magnitudes, ID_statistics, weather, date_sin, date_cos, time in val_dl:
            spike_nums = spike_nums.to(device)
            spike_durations = spike_durations.to(device)
            spike_magnitudes = spike_magnitudes.to(device)
            ID_statistics = ID_statistics.to(device)
            weather = weather.to(device)
            date_sin = date_sin.to(device)
            date_cos = date_cos.to(device)
            time = time.to(device)
            outputs = model(torch.cat([ID_statistics, date_sin, date_cos, weather], dim=-1))
            loss = criterion(outputs, spike_nums.squeeze())
            val_loss += loss.item()

    print(f'Epoch {epoch + 1}/{epochs}, Loss: {running_loss / len(train_dl)}, Val Loss: {val_loss / len(val_dl)}')

Epoch 1/100, Loss: 1.5104700308097037, Val Loss: 1.4903043971210936
Epoch 2/100, Loss: 1.4902707192981453, Val Loss: 1.4886650948121416
Epoch 3/100, Loss: 1.486104030149025, Val Loss: 1.4811881057488776
Epoch 4/100, Loss: 1.4849842390470338, Val Loss: 1.4816458506275947
Epoch 5/100, Loss: 1.4846100993323745, Val Loss: 1.4869557155367574
Epoch 6/100, Loss: 1.4834279245451878, Val Loss: 1.4806948484611566
Epoch 7/100, Loss: 1.483120608329773, Val Loss: 1.4877828095314203
Epoch 8/100, Loss: 1.4839237670103709, Val Loss: 1.48165950903456
Epoch 9/100, Loss: 1.4821494262469441, Val Loss: 1.4834483058594166
Epoch 10/100, Loss: 1.482950820107209, Val Loss: 1.4792494178689966
Epoch 11/100, Loss: 1.482103835490712, Val Loss: 1.4800099827246764
Epoch 12/100, Loss: 1.4824723129732567, Val Loss: 1.4810022664858118


KeyboardInterrupt: 